# Стратегия 1: только реальные данные (KITTI train)

In [23]:
import os
import json
import sys
from pathlib import Path

BASE = '/content/drive/MyDrive/3dcv-project'

DATA_DIR = f'{BASE}/data'
KITTI_ROOT = f'{DATA_DIR}/kitti'
VKITTI_ROOT = f'{DATA_DIR}/vkitti2'

CONFIG_DIR = f'{BASE}/results/track_b/configs'
os.makedirs(CONFIG_DIR, exist_ok=True)

SHARED_DIR = f'{BASE}/code/shared'
VKITTI_LOADER_DIR = f'{BASE}/code/shared/vkitti'

sys.path.append(SHARED_DIR)
sys.path.append(VKITTI_LOADER_DIR)

import _loader as vkitti_loader

loader = vkitti_loader.VKITTI2Loader(VKITTI_ROOT)

print("KITTI exists:", os.path.exists(KITTI_ROOT))
print("VKITTI2 exists:", os.path.exists(VKITTI_ROOT))
print("CONFIG_DIR:", CONFIG_DIR)

KITTI exists: True
VKITTI2 exists: True
CONFIG_DIR: /content/drive/MyDrive/3dcv-project/results/track_b/configs


In [24]:
config_real_only = {
    'name': 'real_only',
    'train_images': 'kitti/training/image_2/',
    'depth_source': 'kitti_predicted',
    'description': 'Используем только реальные изображения KITTI с предсказанной глубиной'
}

# Стратегия 2: только синтетика (VKITTI2 clone)

In [25]:
config_synth_only = {
    'name': 'synth_only',
    'train_images': 'vkitti2/Scene01-20/clone/',
    'depth_source': 'vkitti_gt',
    'description': 'Только VKITTI2 clone с истинной глубиной'
}

# Стратегия 3: смешанная (50/50)

In [26]:
config_mixed = {
    'name': 'mixed_50_50',
    'real_ratio': 0.5,
    'synth_ratio': 0.5,
    'real_source': 'kitti/training/image_2/',
    'synth_source': 'vkitti2/Scene01-20/clone/',
    'description': 'Поровну реальных и синтетических данных'
}


In [27]:
configs = [config_real_only, config_synth_only, config_mixed]

configs

[{'name': 'real_only',
  'train_images': 'kitti/training/image_2/',
  'depth_source': 'kitti_predicted',
  'description': 'Используем только реальные изображения KITTI с предсказанной глубиной'},
 {'name': 'synth_only',
  'train_images': 'vkitti2/Scene01-20/clone/',
  'depth_source': 'vkitti_gt',
  'description': 'Только VKITTI2 clone с истинной глубиной'},
 {'name': 'mixed_50_50',
  'real_ratio': 0.5,
  'synth_ratio': 0.5,
  'real_source': 'kitti/training/image_2/',
  'synth_source': 'vkitti2/Scene01-20/clone/',
  'description': 'Поровну реальных и синтетических данных'}]

# Для каждой стратегии создай **списки путей к файлам**, которые мы используем как trainset:

In [31]:
from pathlib import Path

filelists = {}

for config in configs:

    files = []

    # ======================================================
    # REAL ONLY
    # ======================================================

    if config['name'] == 'real_only':

        kitti_dir = Path(KITTI_ROOT) / 'training' / 'image_2'

        files = [
            str(p)
            for p in sorted(kitti_dir.glob('*.png'))
        ]

    # ======================================================
    # SYNTH ONLY
    # ======================================================

    elif config['name'] == 'synth_only':

        for scene in loader.SCENES:

            frame_ids = loader.list_frames(
                scene=scene,
                variation='clone'
            )

            for fid in frame_ids:

                rgb_path = (
                    Path(VKITTI_ROOT)
                    / scene
                    / 'clone'
                    / 'frames'
                    / 'rgb'
                    / 'Camera_0'
                    / f'rgb_{fid}.jpg'
                )

                if rgb_path.exists():
                    files.append(str(rgb_path))

    # ======================================================
    # MIXED 50/50
    # ======================================================

    elif config['name'] == 'mixed_50_50':

        kitti_dir = Path(KITTI_ROOT) / 'training' / 'image_2'

        real_files = [
            str(p)
            for p in sorted(kitti_dir.glob('*.png'))
        ][:3500]

        synth_files = []

        for scene in loader.SCENES:

            frame_ids = loader.list_frames(
                scene=scene,
                variation='clone'
            )[:700]

            for fid in frame_ids:

                rgb_path = (
                    Path(VKITTI_ROOT)
                    / scene
                    / 'clone'
                    / 'frames'
                    / 'rgb'
                    / 'Camera_0'
                    / f'rgb_{fid}.jpg'
                )

                if rgb_path.exists():
                    synth_files.append(str(rgb_path))

        files = real_files + synth_files

    # ======================================================

    filelists[config['name']] = {
        'config': config,
        'files': files,
        'count': len(files)
    }

    print(f'✅ {config["name"]}: {len(files)} файлов')

✅ real_only: 7481 файлов
✅ synth_only: 2126 файлов
✅ mixed_50_50: 5489 файлов


In [34]:
filelists['mixed_50_50']['files'][:5]

['/content/drive/MyDrive/3dcv-project/data/kitti/training/image_2/000000.png',
 '/content/drive/MyDrive/3dcv-project/data/kitti/training/image_2/000001.png',
 '/content/drive/MyDrive/3dcv-project/data/kitti/training/image_2/000002.png',
 '/content/drive/MyDrive/3dcv-project/data/kitti/training/image_2/000003.png',
 '/content/drive/MyDrive/3dcv-project/data/kitti/training/image_2/000004.png']

# Сохраняем конфиги

In [35]:
for name, data in filelists.items():

    output_path = f'{CONFIG_DIR}/{name}.json'

    with open(output_path, 'w') as f:
        json.dump(data, f, indent=2)

    print(f'💾 Saved: {output_path}')

💾 Saved: /content/drive/MyDrive/3dcv-project/results/track_b/configs/real_only.json
💾 Saved: /content/drive/MyDrive/3dcv-project/results/track_b/configs/synth_only.json
💾 Saved: /content/drive/MyDrive/3dcv-project/results/track_b/configs/mixed_50_50.json
